<a href="https://colab.research.google.com/github/siya-pathak/from-scratch/blob/main/decoder_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-03 05:00:46--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-03 05:00:47 (35.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))
print(text[:300])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [5]:
chars = sorted(list(set(text)))
print(''.join(chars))
print(len(chars)) #size of our vocabulary
vocab_size = len(chars)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
#we want to tokenize here
#character level tokenization
stoi = {chi:i for i, chi in enumerate(chars)}
itos = {i:chi for i, chi in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("shakespeare"))
print(decode(encode("shakespeare")))

[57, 46, 39, 49, 43, 57, 54, 43, 39, 56, 43]
shakespeare


In [7]:
#tokenize the entire dataset we have
#we use pytorch to make the vocab into a tensor
import torch
data =torch.tensor(encode(text))
print(data.shape, data.dtype)
print(data[:50])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [8]:
n = int(0.9*(len(data)))
train = data[:n]
val = data[n:]

In [8]:
#we do not simply feed in entire data to transformer -> compuatationally expensive
#we sample random data, chunks

In [9]:
context_len = 10
train[:context_len+1] #this has 10 examples packed in the sense of -> for these input, this is the char that would come next

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64])

In [10]:
x = train[:context_len]
y = train[1:context_len+1] #offset by 1
for t in range(context_len):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}") #transformer should be in the habit of seeing as little as 1 charcater to block_size

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]) the target: 64


In [11]:
#mini batches of multiple chunks of text, stacked up, for efficiency to keep GPUs busy
torch.manual_seed(28309123719)
batch_size = 4 #how many chunks are to be processed in parallel
context_len = 10 #characters to be processed for next prediction

def get_batch(split):
  data = train if split == "train" else val
  ix = torch.randint(len(data) - context_len, (batch_size,)) #this prodcues batch_size number of indexes (4 index because we need to send 4 chunks)
  x = torch.stack([data[i:i+context_len] for i in ix])
  y = torch.stack([data[i+1: i+context_len+1] for i in ix]) #torch.randint's upper bound is exclusive
  #if len(data) = 20 and context len is 10 max we can get is between 0 and 9 so if i = 9, we index 9 to 9 + 10 which is 10 numbers and then we do 10 to 20
  return x, y

xb, yb = get_batch("train")
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
#the row vectors are the chunks we send for training, 4 rows of 10 columns

inputs:
torch.Size([4, 10])
tensor([[51, 40, 50, 43, 57, 10,  1, 39, 58,  1],
        [58,  6,  1, 44, 56, 43, 43, 50, 63,  0],
        [41, 58,  1, 53, 44,  1, 50, 53, 60, 43],
        [ 0, 31, 58, 39, 63,  1, 52, 53, 58,  1]])
targets:
torch.Size([4, 10])
tensor([[40, 50, 43, 57, 10,  1, 39, 58,  1, 57],
        [ 6,  1, 44, 56, 43, 43, 50, 63,  0, 32],
        [58,  1, 53, 44,  1, 50, 53, 60, 43,  1],
        [31, 58, 39, 63,  1, 52, 53, 58,  1, 58]])


In [12]:
for b in range(batch_size): # batch dimension - 4
    for t in range(context_len): # time dimension - 10
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [51] the target: 40
when input is [51, 40] the target: 50
when input is [51, 40, 50] the target: 43
when input is [51, 40, 50, 43] the target: 57
when input is [51, 40, 50, 43, 57] the target: 10
when input is [51, 40, 50, 43, 57, 10] the target: 1
when input is [51, 40, 50, 43, 57, 10, 1] the target: 39
when input is [51, 40, 50, 43, 57, 10, 1, 39] the target: 58
when input is [51, 40, 50, 43, 57, 10, 1, 39, 58] the target: 1
when input is [51, 40, 50, 43, 57, 10, 1, 39, 58, 1] the target: 57
when input is [58] the target: 6
when input is [58, 6] the target: 1
when input is [58, 6, 1] the target: 44
when input is [58, 6, 1, 44] the target: 56
when input is [58, 6, 1, 44, 56] the target: 43
when input is [58, 6, 1, 44, 56, 43] the target: 43
when input is [58, 6, 1, 44, 56, 43, 43] the target: 50
when input is [58, 6, 1, 44, 56, 43, 43, 50] the target: 63
when input is [58, 6, 1, 44, 56, 43, 43, 50, 63] the target: 0
when input is [58, 6, 1, 44, 56, 43, 43, 50, 63, 0] the

In [13]:
print(xb)
print(yb)

tensor([[51, 40, 50, 43, 57, 10,  1, 39, 58,  1],
        [58,  6,  1, 44, 56, 43, 43, 50, 63,  0],
        [41, 58,  1, 53, 44,  1, 50, 53, 60, 43],
        [ 0, 31, 58, 39, 63,  1, 52, 53, 58,  1]])
tensor([[40, 50, 43, 57, 10,  1, 39, 58,  1, 57],
        [ 6,  1, 44, 56, 43, 43, 50, 63,  0, 32],
        [58,  1, 53, 44,  1, 50, 53, 60, 43,  1],
        [31, 58, 39, 63,  1, 52, 53, 58,  1, 58]])


### Why do we sample from a Multinomial Distribution?

In our `BigramLanguageModel`, we use `torch.multinomial(probs, num_samples=1)` to pick the next character rather than just picking the character with the highest probability (greedy search / `argmax`). Here are the key reasons why:

1. **Diversity and Creativity in Generation**:
   - If we always pick the character with the absolute highest probability, the model becomes completely deterministic. Given the exact same starting context, it will generate the exact same text every single time.
   - Sampling introduces controlled randomness, allowing the model to generate different, diverse variations on each run.

2. **Avoiding Infinite Loops (Repetitive Text)**:
   - Purely greedy selection often gets stuck in infinite loops. For example, it might generate: `"The king went to the court and the court and the court and..."` because "and the court" recursively becomes the most probable next sequence.
   - Sampling breaks these repetitive cycles because less probable (but still highly valid) paths can be chosen.

3. **Modeling Human Language Probabilities**:
   - Human writing and speech are inherently varied. When we write, we don't always choose the most statistically "common" word next; we choose from a pool of appropriate words. Sampling reflects this natural distribution.



* **Multinomial** is designed specifically for **discrete, mutually exclusive, categorical outcomes** where the number of categories $K > 2$. Since selecting a token from a vocabulary of size $V$ is exactly a 1-of-$V$ categorical choice, the multinomial distribution is the mathematically correct model.

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(28309123719)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
  # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    #idx, targets or xb and yb are tensors of tensors of integers -> 2d (B, T)
    logits = self.token_embedding_table(idx) #embeddings of the next token being any of the 65 token in vocab this logit is in the 3rd dimension
    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    #loss = -log(p(y)) = -log(1/65) #currently all at 1/65
    return logits, loss

  def generate(self, idx, max_new_tokens): #idx is (B, T) -> (Batch, Time) -> (Chunk, Context)
    for _ in range(max_new_tokens):
      logits, loss = self(idx) #logits are (B, T, C)
      #focus on last time step and get the full batch's last time step value and the 65 values
      logits = logits[:,-1,:] #(B, C)
      probability = F.softmax(logits, dim=1) #we need to have probability of the 65 values we have
      idx_next = torch.multinomial(probability, num_samples=1) #(B, 1) for each of the batches we get something
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim = 1) #along the time dimension or the context direction we have more and more generation coming up
      #generation from the model, will be B, T+max_new_tokens
    return idx
#bigram makes no sense, we only give the last character to predict the next one,
#we should have more better context like going through everything that has been produced, history
m = BigramLanguageModel(vocab_size=65)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([40, 65])
tensor(4.7721, grad_fn=<NllLossBackward0>)


In [15]:
idx = torch.zeros((1, 1), dtype=torch.int64) # kick off the generation with long integer dtype
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


r;rEnTrozhC IjGBK;AKyG
wJPkoWfw
EF3V,oBwew
xdmDX'SeUAsr'YAIrHM:f,iVQFYxOrbc3ahgKDWdv&HTGrriRqtsgMvuX


In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-5) #3e-4 typically good but our network small

In [17]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #sets gradients to none instead of 0, less memory overhead
    loss.backward()
    optimizer.step()
print(loss.item())

4.511425018310547


In [18]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))
#we give 1 batch and 1 time sequence input and we expect 500 next tokens for the 1 batch giivng (1, 501) output so we index into that one row


'UJYCWIBloqAxOLGxOGBE.byewJ PBrFYHxg;X&.K3fNctnhUahiY3P;rlK
JgiGk;ejbyb.b:JEQXVQa;fDX,lsP.ndxWe-vlAGs.:kTFmcIX'V.nUM!pjvxs SozUR;STA&-brHSspFXgY:$;aUD DZxk&.befwIu&sLcAnsRM!?Y-pS3WFiiS;Mj$HOvxMC,oZD!-UrnUUA piW D&wAkYxCuqOTGBS
Ix'EC$3QvEnnjOoZDWICtLE3sHaJETJqggkuSMsU IAzsFmEA DP.p;rxtWJ
MYoeaIeLvvxOvLIJDztpJvhGZFd3qtP&NbzPlIw.nF
;XJP3NR$;N,ovbs!
AreN;r q.yYzxVQ !pQBNkrd3xgVCARHZutHxOuw
ElAUP!p&f-vby3QTThC$OoZQO:rH;fdvnUVPZllaGsmq$e
w
IS-ozxEdC$TEn'UyDNA !vahpbaIsQBzgQOHpDQnV
kNkFbUrdvx:e:iLPRUUW


In [19]:
torch.manual_seed(28309123719)
B, T, C = 4,10,2
x = torch.randn(B,T,C) #we want these 10 tokens to talk to each other(never future token),
#instead of a plain bigram model looking at just the last letter
#info flows from previous context only
#average channel value of all the previous time step and at this token
x.shape

torch.Size([4, 10, 2])

In [20]:
xbow = torch.zeros((B,T,C)) #bag of words
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] # (t,C) upto and including the Tth token
    xbow[b,t] = torch.mean(xprev, 0) #averaging out the time

In [21]:
x[0] #first batch

tensor([[ 0.3045, -1.0790],
        [ 1.6964, -1.0390],
        [ 1.2687,  0.7044],
        [-0.1122, -0.0758],
        [-0.0668, -0.4321],
        [ 0.2288,  0.1591],
        [ 0.2073,  1.8417],
        [ 0.0950, -1.0245],
        [-2.1145, -0.1788],
        [ 0.8639,  1.5299]])

In [22]:
xbow[0] #each of the channels, value at i is average of all values from 0 to i

tensor([[ 0.3045, -1.0790],
        [ 1.0005, -1.0590],
        [ 1.0899, -0.4712],
        [ 0.7894, -0.3724],
        [ 0.6181, -0.3843],
        [ 0.5532, -0.2937],
        [ 0.5038,  0.0113],
        [ 0.4527, -0.1182],
        [ 0.1675, -0.1249],
        [ 0.2371,  0.0406]])

In [23]:
#the above is inefficient
#matrix multiplication can help

In [24]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(28309123719)
a = torch.tril(torch.ones(3, 3)) #lower triangular matrix #helps keep a running average like we want
#(a+b+c)/3 is (1/3)a+(1/3)b+(1/3)c
a = a/torch.sum(a, 1, keepdim=True) #normalization across columns
b = torch.randint(0,10,(3,2)).float()
c = a @ b #matrix multiplication
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[7., 2.],
        [0., 5.],
        [4., 8.]])
--
c=
tensor([[7.0000, 2.0000],
        [3.5000, 3.5000],
        [3.6667, 5.0000]])


In [25]:
#using matrix multiply for a weighted aggregation
weight = torch.tril(torch.ones(T, T)) #lower triangular
weight = weight / weight.sum(1, keepdim=True)
xbow2 = weight @ x # (B, T, T) @ (B, T, C) ----> (B, T, C) #across the B we will have (T, T) @ (T, C) -> (T, C)
torch.allclose(xbow, xbow2)

True

In [26]:
tril = torch.tril(torch.ones(T, T))
weight = torch.zeros((T, T))
weight = weight.masked_fill(tril == 0, float('-inf')) #future can't communicate with the past
weight = F.softmax(weight, dim=1) #softmax is also a normalization opertaion only
xbow3 = weight@x
torch.allclose(xbow, xbow3)

True

In [35]:
torch.manual_seed(28309123719)
B, T, C = 4,10,32
x = torch.randn(B,T,C) #batch, time, channels

#single head performing attention
#head size is the size or width of the query (Q), key (K), and value (V) vectors inside a single attention head.

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
key = key(x) #B, T, 16
query = query(x) #B, T, 16
weight = query @ key.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)
#these variance of weights are of the order of head size
#add scaled attention factor
#weight is given to softmax so we want it fairly diffused
#weight if very positive or very negative, softmax move to 1,-1 (one hot almost)

tril = torch.tril(torch.ones(T, T))
weight = weight.masked_fill(tril == 0, float('-inf')) #future can't communicate with the past
weight = F.softmax(weight, dim=1) #softmax is also a normalization opertaion only

v = value(x) #x is private information token
#this is what i am looking for(queries), this is what we have(keys), this is what we can communicate(v)
out = weight @ v
#attention does not really care for whether we talk with future or past, it can do all
#in encoder we do not mask_fill
#in decoder we do to keep the autoregressive nature alive
#self attention is called such because the keys, queries, values are all being produced from the same source
#cross attention, queries from x, keys and values from different source(like the encoder block)
out.shape

torch.Size([4, 10, 16])

In [32]:
weight[0]

tensor([[0.0089, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0667, 0.0622, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.3165, 0.0523, 0.0058, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0495, 0.3460, 0.1284, 0.0833, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0390, 0.2203, 0.1613, 0.0620, 0.0507, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1216, 0.2137, 0.1062, 0.2702, 0.0057, 0.1388, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0034, 0.0030, 0.2649, 0.0270, 0.6344, 0.1742, 0.0658, 0.0000, 0.0000,
         0.0000],
        [0.3200, 0.0063, 0.0359, 0.2124, 0.0101, 0.1291, 0.3528, 0.8882, 0.0000,
         0.0000],
        [0.0122, 0.0064, 0.0434, 0.0473, 0.0345, 0.4533, 0.0201, 0.0723, 0.1028,
         0.0000],
        [0.0620, 0.0897, 0.2541, 0.2979, 0.2646, 0.1045, 0.5613, 0.0395, 0.8972,
         1.0000]], grad_fn=<

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(28309123719)
n_embd = 32 #embedding size
device = 'cuda' if torch.cuda.is_available() else 'cpu'
context_len = 32
batch_size = 16
max_iters = 10000
eval_interval = 100
eval_iters = 200

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train = data[:n]
val = data[n:]

def get_batch(split):
  data = train if split == "train" else val
  ix = torch.randint(len(data) - context_len, (batch_size,)) #this prodcues batch_size number of indexes (4 index because we need to send 4 chunks)
  x = torch.stack([data[i:i+context_len] for i in ix])
  y = torch.stack([data[i+1: i+context_len+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    #tril is a fixed tensor that the model needs, but it is not something we want to train.
    self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))

  def forward(self, x):
    B, T, C = x.shape
    key = self.key(x) #B, T, C
    query = self.query(x) #B, T, C
    weight = query @ key.transpose(-2, -1) # (B, T, C) @ (B, C, T) ---> (B, T, T)
    weight *= key.shape[-1] ** -0.5
    weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #future can't communicate with the past
    weight = F.softmax(weight, dim=-1) #softmax is also a normalization opertaion only
    value = self.value(x)
    out = weight @ value
    return out #B, T, T @ B, T, C -> B, T, C


class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

  def forward(self, x):
    return torch.cat([h(x) for h in self.heads], dim = -1)

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
  # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.sa_head = MultiHeadAttention(4, n_embd//4)
    self.lm_head = nn.Linear(n_embd, vocab_size)
    self.position_embedding_table = nn.Embedding(context_len, n_embd) #each of the letters in the context length also get an embedding vector

  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) #B,T,C=32
    # Get position numbers 0, 1, 2, ..., T-1
    # This gives one position embedding for each time step
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #T, C
    #tok_emb = (B, T, C)
    #pos_emb = (T, C) -> added across all batches, broadcasting
    #when matching tensor shapes for broadcasting, PyTorch compares dimensions from right to left.
    x = tok_emb + pos_emb
    x = self.sa_head(x)
    logits = self.lm_head(x) #B, T, vocab_size

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    #loss = -log(p(y)) = -log(1/65) #currently all at 1/65
    return logits, loss

  def generate(self, idx, max_new_tokens): #idx is (B, T) -> (Batch, Time) -> (Chunk, Context)
    for _ in range(max_new_tokens):
      # The model can only handle up to context_len tokens at a time,
      # so we keep only the most recent context_len tokens as context.
      #will cause issues in positional embeddings and tril
      cropped_index = idx[:, -context_len:]
      logits, loss = self(cropped_index)
      logits = logits[:,-1,:] #(B, C)
      probability = F.softmax(logits, dim=1)
      idx_next = torch.multinomial(probability, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

model = BigramLanguageModel().to(device)


--2026-10-03 05:26:46--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-10-03 05:26:47 (195 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3) #3e-4 typically good but our network small

In [3]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.1915, val loss 4.1946
step 100: train loss 3.1867, val loss 3.2169
step 200: train loss 2.9807, val loss 3.0042
step 300: train loss 2.8452, val loss 2.8595
step 400: train loss 2.7835, val loss 2.7836
step 500: train loss 2.7248, val loss 2.7301
step 600: train loss 2.6927, val loss 2.6960
step 700: train loss 2.6507, val loss 2.6579
step 800: train loss 2.6339, val loss 2.6407
step 900: train loss 2.6199, val loss 2.6263
step 1000: train loss 2.6048, val loss 2.6138
step 1100: train loss 2.5920, val loss 2.5946
step 1200: train loss 2.5774, val loss 2.5818
step 1300: train loss 2.5639, val loss 2.5696
step 1400: train loss 2.5537, val loss 2.5525
step 1500: train loss 2.5391, val loss 2.5431
step 1600: train loss 2.5284, val loss 2.5365
step 1700: train loss 2.5208, val loss 2.5158
step 1800: train loss 2.5051, val loss 2.5077
step 1900: train loss 2.4969, val loss 2.4945
step 2000: train loss 2.4929, val loss 2.4893
step 2100: train loss 2.4762, val loss 2.4787


In [5]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass

    xmean = x.mean(1, keepdim=True) # layer mean, mean across features of the vector
    xvar = x.var(1, keepdim=True) # layer variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(28309123719)
n_embd = 32 #embedding size
device = 'cuda' if torch.cuda.is_available() else 'cpu'
context_len = 32
batch_size = 16
max_iters = 10000
eval_interval = 100
eval_iters = 200

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train = data[:n]
val = data[n:]

def get_batch(split):
  data = train if split == "train" else val
  ix = torch.randint(len(data) - context_len, (batch_size,)) #this prodcues batch_size number of indexes (4 index because we need to send 4 chunks)
  x = torch.stack([data[i:i+context_len] for i in ix])
  y = torch.stack([data[i+1: i+context_len+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Block(nn.Module):
  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size) #communication
    self.ffwd = FeedForward(n_embd) #computation
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

#each token's 32 numbers ->Linear layer -> 32 new numbers -> ReLU -> 32 numbers
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, 4 * n_embd), #4 comes from the paper of attention
      nn.ReLU(),
      nn.Linear(4 * n_embd, n_embd)
    )

  def forward(self, x):
        return self.net(x)

class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    #tril is a fixed tensor that the model needs, but it is not something we want to train.
    self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))

  def forward(self, x):
    B, T, C = x.shape
    key = self.key(x) #B, T, C
    query = self.query(x) #B, T, C
    weight = query @ key.transpose(-2, -1) # (B, T, C) @ (B, C, T) ---> (B, T, T)
    weight *= key.shape[-1] ** -0.5
    weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #future can't communicate with the past
    weight = F.softmax(weight, dim=-1) #softmax is also a normalization opertaion only
    value = self.value(x)
    out = weight @ value
    return out #B, T, T @ B, T, C -> B, T, C


class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd )

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.proj(out)
    return out

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
  # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(context_len, n_embd) #each of the letters in the context length also get an embedding vector
    self.blocks = nn.Sequential(
      Block(n_embd, n_head =4),
      Block(n_embd, n_head =4),
      Block(n_embd, n_head =4),
      nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(n_embd, vocab_size)


  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) #B,T,C=32
    # Get position numbers 0, 1, 2, ..., T-1
    # This gives one position embedding for each time step
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #T, C
    #tok_emb = (B, T, C)
    #pos_emb = (T, C) -> added across all batches, broadcasting
    #when matching tensor shapes for broadcasting, PyTorch compares dimensions from right to left.
    x = tok_emb + pos_emb #B, T, C
    x = self.blocks(x) #B, T, C
    logits = self.lm_head(x) #B, T, vocab_size

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    #loss = -log(p(y)) = -log(1/65) #currently all at 1/65
    return logits, loss

  def generate(self, idx, max_new_tokens): #idx is (B, T) -> (Batch, Time) -> (Chunk, Context)
    for _ in range(max_new_tokens):
      # The model can only handle up to context_len tokens at a time,
      # so we keep only the most recent context_len tokens as context.
      #will cause issues in positional embeddings and tril
      cropped_index = idx[:, -context_len:]
      logits, loss = self(cropped_index)
      logits = logits[:,-1,:] #(B, C)
      probability = F.softmax(logits, dim=1)
      idx_next = torch.multinomial(probability, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

model = BigramLanguageModel().to(device)

--2026-10-03 06:58:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-10-03 06:58:27 (42.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3) #3e-4 typically good but our network small

In [3]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2534, val loss 4.2564
step 100: train loss 2.9775, val loss 2.9943
step 200: train loss 2.7109, val loss 2.7211
step 300: train loss 2.5956, val loss 2.6066
step 400: train loss 2.5324, val loss 2.5425
step 500: train loss 2.4739, val loss 2.4815
step 600: train loss 2.4378, val loss 2.4363
step 700: train loss 2.4081, val loss 2.4091
step 800: train loss 2.3813, val loss 2.3894
step 900: train loss 2.3574, val loss 2.3677
step 1000: train loss 2.3326, val loss 2.3414
step 1100: train loss 2.3051, val loss 2.3184
step 1200: train loss 2.2896, val loss 2.3034
step 1300: train loss 2.2687, val loss 2.2794
step 1400: train loss 2.2414, val loss 2.2590
step 1500: train loss 2.2160, val loss 2.2455
step 1600: train loss 2.2059, val loss 2.2227
step 1700: train loss 2.1922, val loss 2.2143
step 1800: train loss 2.1779, val loss 2.1979
step 1900: train loss 2.1576, val loss 2.1783
step 2000: train loss 2.1509, val loss 2.1740
step 2100: train loss 2.1411, val loss 2.1642


In [5]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))



BRUTUMEY:
Kou have that the our mye thy alows. Campes.

CORTHFRYTES:
I could forround this ruch prerrouse
But o'ght your tine as vasted would
The yeford: upoppance them wall
an in the criad, and eyeincersitor not and vick
thue of it a mare to in wlould a set it grave:
Wast cannot meant yefoul: bo atchoofthouch his acklace.

Praporsome!
When live thou the like. And
Buher the's no vaition if after talt.
But hast but the mailt seem poorcevedse attians. Ahabare
In truck heave you squeen, I gode
A have neat, our and your asame,
To.

HENRMANLEN
Secont.
Swears ret more to the of and frety ludion fromom it,
There frove. For do, that smard:
I knall a ben the dues! deter,
Thouse elpeds, and mandted, made yecarling tordiesion,
The sake pantoods: allo to lick Bubain is heads own alior;
An fouspeod in if like; he's men
Asisuy to ried.
fathers: bair my yourse: fays
That by bear, a seedtry wear a game a to the port, falll;
When, not gixtence law's the of wiellow?

VABENCE:
Say 'Tis chich heive and 

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(28309123719)
n_embd = 384 #embedding size
device = 'cuda' if torch.cuda.is_available() else 'cpu'
context_len = 256
batch_size = 64
max_iters = 10000
eval_interval = 500
eval_iters = 200
n_head = 6
n_layers = 6
dropout = 0.2
lr = 3e-4

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train = data[:n]
val = data[n:]

def get_batch(split):
  data = train if split == "train" else val
  ix = torch.randint(len(data) - context_len, (batch_size,)) #this prodcues batch_size number of indexes (4 index because we need to send 4 chunks)
  x = torch.stack([data[i:i+context_len] for i in ix])
  y = torch.stack([data[i+1: i+context_len+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Block(nn.Module):
  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size) #communication
    self.ffwd = FeedForward(n_embd) #computation
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

#each token's 32 numbers ->Linear layer -> 32 new numbers -> ReLU -> 32 numbers
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, 4 * n_embd), #4 comes from the paper of attention
      nn.ReLU(),
      nn.Linear(4 * n_embd, n_embd),
      nn.Dropout(dropout)
    )

  def forward(self, x):
        return self.net(x)

class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    #tril is a fixed tensor that the model needs, but it is not something we want to train.
    self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape
    key = self.key(x) #B, T, C
    query = self.query(x) #B, T, C
    weight = query @ key.transpose(-2, -1) # (B, T, C) @ (B, C, T) ---> (B, T, T)
    weight *= key.shape[-1] ** -0.5
    weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #future can't communicate with the past
    weight = F.softmax(weight, dim=-1) #softmax is also a normalization opertaion only
    weight = self.dropout(weight)
    value = self.value(x)
    out = weight @ value
    return out #B, T, T @ B, T, C -> B, T, C


class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd )
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
  # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(context_len, n_embd) #each of the letters in the context length also get an embedding vector
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layers)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)


  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) #B,T,C=32
    # Get position numbers 0, 1, 2, ..., T-1
    # This gives one position embedding for each time step
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #T, C
    #tok_emb = (B, T, C)
    #pos_emb = (T, C) -> added across all batches, broadcasting
    #when matching tensor shapes for broadcasting, PyTorch compares dimensions from right to left.
    x = tok_emb + pos_emb #B, T, C
    x = self.blocks(x) #B, T, C
    x = self.ln_f(x) # (B,T,C)
    logits = self.lm_head(x) #B, T, vocab_size

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    #loss = -log(p(y)) = -log(1/65) #currently all at 1/65
    return logits, loss

  def generate(self, idx, max_new_tokens): #idx is (B, T) -> (Batch, Time) -> (Chunk, Context)
    for _ in range(max_new_tokens):
      # The model can only handle up to context_len tokens at a time,
      # so we keep only the most recent context_len tokens as context.
      #will cause issues in positional embeddings and tril
      cropped_index = idx[:, -context_len:]
      logits, loss = self(cropped_index)
      logits = logits[:,-1,:] #(B, C)
      probability = F.softmax(logits, dim=1)
      idx_next = torch.multinomial(probability, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

model = BigramLanguageModel().to(device)

--2026-10-03 07:09:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-03 07:09:26 (32.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) #3e-4 typically good but our network small

In [3]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.3285, val loss 4.3245
step 500: train loss 1.9175, val loss 2.0175
step 1000: train loss 1.5512, val loss 1.7373


KeyboardInterrupt: 

In [4]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))


ISABEL:
I thou witht ingrave 'gain, pend them
And came! I thine you it.

KING HENRY VI:
Hesinus in bink, 'Was pueen gentlive,
And is himsell, and hold spro't.

GREEMO:
Id is do this sward
By trutle anvery, cridio love mangty?

MERLANE:
Now that earten and for this one the frurry.

WIRG KIDWARD CHARD:
Ten me of mearl woman to my till any?
Forst my your RiCK:
Marry by plervind from my heave iffard?

MONTAGUCE:
Pronsed, I am ways loven so righalighn beaAnd you,--

ROMEO:
Sord talk, my viry see to that fores gunt:--

LUCENTIURGA:
I'll to them, dillines swoets resay to me,
'd as if he I brlove suft,
Hoping looke to rousagainst his bast,
Not, and my till gepare.

ELLO:
You shall dount whith May you,
Dest what is, true thee lady.
All, and his offore mile hef.

AUULET:
I'll Amnst you dist.

QUEEN ELIZABETZABETH:
Nir he word, till caletes me,
Whochut so you, inglard you shall if in your would
Work you? bele lorde, harm for so yet
riss that, I sagge at are on't the not
Thas.
Then, teres Come, t